# Introduction to Boltzmann Machines
In this module, we will explore Boltzmann Machines, a type of stochastic recurrent neural network. Boltzmann machines come in two forms: full Boltzmann machines and restricted Boltzmann machines. The latter is a simplified version that is easier to train and is often used in practice. By the end of this module, you will be able to define and demonstrate mastery of the following key concepts:
* __Boltzmann machines__ are stochastic, fully connected networks of binary units with symmetric weights, which define a Boltzmann probability distribution over its states. It learns to model complex data distributions by adjusting its weights, so that the network’s sampled statistics match the training data.
* __Restricted Boltzmann machines (RBMs)__ are a simplified version of Boltzmann machines with two layers: a visible layer and a hidden layer. The visible layer represents the input data, while the hidden layer captures the underlying structure of the data. RBMs are easier to train and are often used in practice.

<div>
    <center>
      <img
        src="figs/Fig-Boltzmann-Machine-Schematic.svg"
        alt="triangle with all three sides equal"
        height="400"
        width="800"
      />
    </center>
  </div>

## Boltazmann Machines
A [Boltzmann Machine](https://en.wikipedia.org/wiki/Boltzmann_machine) consists of a set of binary units (neurons, nodes, vertices, etc.) that are fully connected, with no self-connections. 

* _Nodes in a Boltzmann machine_? Each node can be in one of two states: `on` or `off.` The units have bias terms and are connected to every other node in the network with weighted edges. The state of each unit is determined by the states of the other units and the weights of the connections. The state of a node is a random variable.

Formally, [a Boltzmann Machine](https://en.wikipedia.org/wiki/Boltzmann_machine) $\mathcal{B}$ is an fully connected _undirected weighted graph_ defined by the tuple $\mathcal{B} = \left(\mathcal{V},\mathcal{E}, \mathbf{W},\mathbf{b}, \mathbf{s}\right)$.
* __Units__: Each unit (vertex, node, neuron) $v_{i}\in\mathcal{V}$ has a binary state (`on` or `off`) and a bias value 
$b_{i}\in\mathbb{R}$. The bias vector $\mathbf{b}\in\mathbb{R}^{|\mathcal{V}|}$ is the vector of bias values for all nodes in the network. 
    - _Visible versus hidden_: A network may have _visible_ nodes (state is visible), and _hidden_ nodes (state is not visible to us). The visible nodes represent the input data, while the hidden nodes capture the underlying structure of the data (latent variables).
    
    The set of all nodes is denoted by $\mathcal{V} \equiv \left\{v_{1},v_{2},\ldots,v_{|\mathcal{V}|}\right\}$, where $|\mathcal{V}|$ is the number of nodes in the network. We can partition the set of nodes into visible nodes $\mathcal{V}_{\text{vis}}$ and hidden nodes $\mathcal{V}_{\text{hid}}$, such that $\mathcal{V} = \mathcal{V}_{\text{vis}} \cup \mathcal{V}_{\text{hid}}$ and $\mathcal{V}_{\text{vis}} \cap \mathcal{V}_{\text{hid}} = \emptyset$.
* __Edges__: Each edge $e\in\mathcal{E}$ has a weight. The weight of the edge connecting $v_{i}\in\mathcal{V}$ and $v_{j}\in\mathcal{V}$, is denoted by $w_{ij}\in\mathbf{W}$, where the weight matrix $\mathbf{W}\in\mathbb{R}^{|\mathcal{V}|\times|\mathcal{V}|}$ is symmetric, i.e. $w_{ij} = w_{ji}$ and $w_{ii} = 0$ (no self loops). The weights $w_{ij}\in\mathbb{R}$ determine the strength of the connection between the two nodes. 
* __States__: The state of each node is represented by a binary vector $\mathbf{s}\in\mathbb{R}^{|\mathcal{V}|}$, where $s_{i}\in\{-1,1\}$ is the state of node $v_{i}$. When $s_{i} = 1$, the node is `on`, and when $s_{i} = -1$, the node is `off`. The set of all possible state _configurations_ is denoted by $\mathcal{S} \equiv \left\{\mathbf{s}^{(1)},\mathbf{s}^{(2)},\ldots,\mathbf{s}^{(N)}\right\}$, where $N$ is the number of possible state configurations, or $N = 2^{|\mathcal{V}|}$ for binary units.

### Stochastic Dynamics
Suppose we let the state of the Boltzmann Machine $\mathcal{B}$ evolve over $t=1,2,\dots, T$ turns. During each turn, every node can update its state based on the states of the other nodes it is connected to, the weights of its connections, and its bias term. The total input to node $v_{i}$ at turn $t$ denoted as $h_{i}^{(t)}$ is given by:
$$
h_{i}^{(t)} = \sum_{j\in\mathcal{V}} w_{ij}s_{j}^{(t-1)} + b_{i}\quad\forall i\in\mathcal{V}
$$
where $w_{ij}$ is the weight of the edge connecting $v_{i}$ and $v_{j}$, and $s_{j}^{(t-1)}$ is the state of node $v_{j}$ at turn $t-1$. However, unlike [classical Hopfield networks](https://en.wikipedia.org/wiki/Hopfield_network), where the update is deterministic, in a Boltzmann Machine, the state of each node is updated stochastically. The probability that node $v_{i}$ is `on` at turn $t$ is given by the logistic function:
$$
P(s_{i}^{(t)} = 1|h_{i}^{(t)}) = \frac{1}{1+\exp(-2\beta{h}_{i}^{(t)})}
$$
where $P(s_{i}^{(t)} = 1|h_{i}^{(t)})$ is the probability that node $v_{i}$ is `on` at time $t$ given the total input $h_{i}^{(t)}$. The probability that node $v_{i}$ is `off` at time $t$ is given by $P(s_{i}^{(t)} = -1|h_{i}^{(t)}) = 1 - P(s_{i}^{(t)} = 1|h_{i}^{(t)})$,  i.e., one minus the probability that the node is `on`.
* _What is β_? The parameter $\beta$ is the (inverse) temperature parameter that controls the amount of randomness in the system. As $\beta\rightarrow\infty$, the Boltzmann Machine becomes more deterministic; however, as $\beta\rightarrow{0}$, the Boltzmann Machine becomes more random. 

### Sampling a Boltzmann Machine (Gibbs Sampling)
To generate samples from a Boltzmann Machine, let us consider the following algorithm: 

__Initialize__ the weights $\mathbf{W}$ and biases $\mathbf{b}$ of the Boltzmann Machine. Provide an initial state $\mathbf{s}^{(0)}$ of the network, a system (inverse) temperature $\beta$, and the number of turns $T$ to run the sampling algorithm.

For each turn $t=1,2,\dots,T$:
1. For each node $v_{i}\in\mathcal{V}$:
    1. Compute the total input $h_{i}^{(t)}$ to node $v_{i}$ using the expression: $h_{i}^{(t)} = \sum_{j\in\mathcal{V}} w_{ij}s_{j}^{(t-1)} + b_{i}$.
    2. Compute the probability of the _next_ state $s_{i}^{(t)} = 1$ using the logistic function $P(s_{i}^{(t)} = 1|h_{i}^{(t)}) = \left(1+\exp(-2\beta{h}_{i}^{(t)})\right)^{-1}$ for node $v_{i}$. The probability of $s_{i}^{(t)} = -1$ is given by $P(s_{i}^{(t)} = -1|h_{i}^{(t)}) = 1 - P(s_{i}^{(t)} = 1|h_{i}^{(t)})$.
    3. Sample the _next_ state of node $v_{i}$ from a [Bernoulli distribution](https://en.wikipedia.org/wiki/Bernoulli_distribution) with parameter $p = P(s_{i}^{(t)} = 1|h_{i}^{(t)})$.
2. Store the state vector $\mathbf{s}^{(t)}$ of the network at turn $t$, and proceed to the next turn.

### Stationary Distribution
For simplicity, let's assume all the nodes are visible nodes, i.e., $\mathcal{V} = \mathcal{V}_{\text{vis}}$ and $\mathcal{V}_{\text{hid}} = \emptyset$. The Boltzmann Machine can be thought of as a stochastic process that evolves over time. The state of the network at each turn is a random variable, and the network can be in one of many possible configurations (state vectors) $\mathbf{s}\in\mathcal{S}$.

After a _sufficiently large_ number of turns $T$, the network configurations (state vectors) $\mathbf{s}^{(1)},\mathbf{s}^{(2)},\dots,$ of the Boltzmann Machine will converge to a _stationary distribution_ over the state configurations $\mathbf{s}\in\mathcal{S}$ which can be modeled as [a Boltzmann distribution](https://en.wikipedia.org/wiki/Boltzmann_distribution) of the form:
$$
P(\mathbf{s}) = \frac{1}{Z(\mathcal{S},\beta)}\exp\left(-\beta\cdot{E(\mathbf{s})}\right)
$$
where $E(\mathbf{s})$ is the energy of state $\mathbf{s}$, the $\beta$ is the (inverse) temperature of the system, and $Z(\mathcal{S},\beta)$ is the partition function. The energy of configuration $\mathbf{s}\in\mathcal{S}$ is given by:
$$
E(\mathbf{s}) = -\sum_{i\in\mathcal{V}} b_{i}s_{i} - \frac{1}{2}\sum_{i,j\in\mathcal{V}} w_{ij}s_{i}s_{j}
$$
where the first term is the energy associated with the bias terms, and the second term is the energy associated with the weights of the connections. The partition function $Z(\mathcal{S},\beta)$ is given by:
$$
Z(\mathcal{S},\beta) = \sum_{\mathbf{s}^{\prime}\in\mathcal{S}}\exp\left({-\beta\cdot{E}(\mathbf{s}^{\prime})}\right)
$$
where $\mathcal{S}$ is the set of _all possible network configurations_ of the Boltzmann Machine. 
* __Hmmm...__? The partition function $Z(\mathcal{S},\beta)$ is a normalizing constant ensuring the probabilities sum to `1`. However, for even a moderately sized system, the partition function is impossible to compute; the number of configurations grows exponentially with the number of nodes. For example, for a $28\times{28}$ image (784 nodes, all visible), the number of possible configurations (states) of the Boltzmann Machine is `2^{784}`.

If we can't compute the partition function, how do we determine if the network is in a stationary distribution? 